## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [8]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Membuat SparkSession & Dataset
Membuat SparkSession & Dataset agar cell selanjutnya dapat dijalankan dengan baik

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/17 13:45:48 WARN Utils: Your hostname, kyadevi resolves to a loopback address: 127.0.1.1; using 10.54.128.92 instead (on interface wlp0s20f3)
26/09/17 13:45:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 13:45:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/17 13:45:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


### Simpan data_target_cabang

In [7]:
import pandas as pd

# Dictionary data target cabang — copy-paste dari modul
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Langsung jadikan Spark DataFrame
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



1. Baca `transaksi_tugas5.csv` dari HDFS menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).

In [12]:
from pyspark.sql.functions import col

# ===== 1. Baca dari HDFS =====
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

print("Schema sebelum tambah pendapatan:")
df_transaksi.printSchema()

# ===== 2. Tambah kolom pendapatan =====
df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

print("\nSchema setelah tambah pendapatan:")
df_transaksi.printSchema()

print("\n5 baris pertama:")
df_transaksi.show(5)

print(f"\nTotal baris: {df_transaksi.count()}")

Schema sebelum tambah pendapatan:
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)


Schema setelah tambah pendapatan:
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)


5 baris pertama:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|    

### Buat `df_target` dari dictionary `data_target_cabang` di atas

In [13]:
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("\nSchema df_target:")
df_target.show()


Schema df_target:
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

In [17]:
# ===== A. JOIN & PERBANDINGAN TARGET =====
from pyspark.sql.functions import round as spark_round

# Langkah 1: Agregasi total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Join dengan df_target
hasil_A = ringkasan_kota.join(df_target, on="kota", how="inner")

# Langkah 3: Hitung pencapaian_persen
hasil_A = hasil_A.withColumn(
    "pencapaian_persen",
    spark_round((col("total_pendapatan") / col("target_bulanan") * 100), 2)
)

# Langkah 4: Urutkan dari pencapaian tertinggi
hasil_A = hasil_A.select(
    "kota", "pic_cabang", "total_pendapatan", 
    "target_bulanan", "pencapaian_persen"
).orderBy(col("pencapaian_persen").desc())

print("=== A. Pencapaian Target per Cabang ===")
hasil_A.show()

=== A. Pencapaian Target per Cabang ===
+----------+----------+----------------+--------------+-----------------+
|      kota|pic_cabang|total_pendapatan|target_bulanan|pencapaian_persen|
+----------+----------+----------------+--------------+-----------------+
| Purworejo|     Fitri|        45650000|      30000000|           152.17|
|      Solo|      Bayu|        33475000|      40000000|            83.69|
|Yogyakarta|      Joko|        47275000|      60000000|            78.79|
|  Magelang|      Rani|        31650000|      45000000|            70.33|
|  Semarang|      Sari|        38175000|      55000000|            69.41|
+----------+----------+----------------+--------------+-----------------+



**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

In [18]:
# ===== B. WINDOW FUNCTION — KATEGORI TERLARIS PER KOTA =====
from pyspark.sql.functions import sum as spark_sum

# Langkah 1: Agregasi pendapatan per (kota, kategori)
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Definisikan window — partisi per kota, urutkan dari pendapatan tertinggi
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Langkah 3: Beri nomor urut dengan row_number()
df_ranked = pendapatan_kategori.withColumn(
    "peringkat", row_number().over(window_kota)
)

# Langkah 4: Ambil hanya peringkat 1 (top-1 per kota)
hasil_B = df_ranked.filter(col("peringkat") == 1) \
    .select("kota", "kategori", "total_pendapatan", "peringkat") \
    .orderBy("kota")

print("=== B. Kategori Terlaris per Kota ===")
hasil_B.show()

=== B. Kategori Terlaris per Kota ===
+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [19]:
# ===== C. SPARK SQL =====

# Daftarkan sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL murni (bukan DataFrame API)
hasil_C = spark.sql('''
    SELECT 
        t.kota,
        g.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target g ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

print("=== C. Jumlah Transaksi per Kota (Spark SQL) ===")
hasil_C.show()

=== C. Jumlah Transaksi per Kota (Spark SQL) ===
+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan** *(bobot 25%)*

Berdasarkan hasil analisis Bagian A (Pencapaian Target per Cabang) dan Bagian B (Kategori Terlaris per Kota), dapat disimpulkan bahwa **cabang dengan kinerja paling baik adalah Purworejo**, yang dipimpin oleh PIC **Fitri**. Cabang ini berhasil membukukan total pendapatan sebesar **Rp45.650.000**, jauh melampaui target bulanan yang ditetapkan sebesar Rp30.000.000, sehingga mencapai **152,17%** — tertinggi di antara seluruh cabang. Kategori andalan di Purworejo adalah **Kesehatan & Kecantikan** dengan total pendapatan **Rp10.075.000**, yang menjadi kontributor terbesar di kota tersebut. Selain unggul dalam pencapaian target, Purworejo juga mencatat **jumlah transaksi terbanyak (116 transaksi)** berdasarkan hasil Bagian C, menunjukkan kombinasi volume dan nilai transaksi yang sehat.

Sebaliknya, **cabang yang paling perlu perhatian manajemen adalah Semarang** (PIC: Sari), karena hanya mencapai **69,41%** dari target bulanan Rp55.000.000, dengan realisasi pendapatan hanya **Rp38.175.000** — kekurangan sebesar **Rp16.825.000**. Meskipun Semarang memiliki kategori andalan **Rumah Tangga** dengan pendapatan **Rp11.125.000**, kontribusi tersebut belum cukup menutupi target yang ditetapkan. Menariknya, jumlah transaksi di Semarang (93 transaksi) justru berada di urutan keempat, lebih tinggi dari Magelang (86 transaksi). Ini mengindikasikan bahwa masalah Semarang **bukan pada jumlah transaksi yang sedikit**, melainkan pada **nilai transaksi per order yang rendah** — kemungkinan karena komposisi produk yang dijual berharga rendah atau strategi diskon yang terlalu agresif. Cabang **Magelang** (PIC: Rani) juga perlu perhatian karena pencapaiannya hanya **70,33%** (realisasi Rp31.650.000 dari target Rp45.000.000), meskipun masih sedikit lebih baik dari Semarang.

**Rekomendasi:**
1. Cabang **Purworejo** dapat dijadikan *benchmark* — praktik terbaiknya pada kategori Kesehatan & Kecantikan dapat direplikasi ke cabang lain, terutama Semarang dan Magelang.
2. Manajemen perlu mengevaluasi strategi pemasaran di **Semarang** dan **Magelang**, khususnya pada aspek *pricing* dan komposisi produk, mengingat keduanya memiliki target tinggi namun realisasi rendah.
3. Perlu analisis lanjutan untuk mengetahui penyebab rendahnya pencapaian di Semarang — apakah karena faktor kompetisi, daya beli, atau strategi promosi yang kurang tepat.

Dengan menggabungkan **join** (Bagian A), **window function** (Bagian B), dan **Spark SQL** (Bagian C), dashboard performa cabang ini dapat dijadikan dasar pengambilan keputusan manajemen secara *data-driven*.